# Airline Passenger Satisfaction - Decision Tree Classification & Hyperparameter Tuning
### 3MTT NextGen Cohort - Step 28 Evaluation

### 1. Data Cleaning, Preprocessing & Feature Encoding
Loading the airline dataset, dropping structural missingness, and encoding non-numeric categorical attributes using pandas dummies.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import confusion_matrix, classification_report, f1_score, accuracy_score
import os

# Locate and load the target dataset
csv_file = [f for f in os.listdir('.') if f.endswith('.csv')][0]
df = pd.read_csv(csv_file)

print("--- Baseline Structural Overview ---")
print(f"Initial Shape: {df.shape}")

# Handle missing data parameters across features securely
df = df.dropna()

# Standardize column layouts and isolate target binary attribute
df.columns = [c.strip() for c in df.columns]

# Identify target column (assuming typical naming convention like 'satisfaction' or similar)
target_col = [c for c in df.columns if 'satisfaction' in c.lower()][0]

# Feature encoding via one-hot transformation mapping for categorical variables
X = df.drop(columns=[target_col])
y = df[target_col].apply(lambda x: 1 if 'satisfied' in str(x).lower() else 0)

X_encoded = pd.get_dummies(X, drop_first=True)
print(f"Processed Features Shape (after dummy encoding): {X_encoded.shape}")

### 2. Hyperparameter Optimization via GridSearchCV
Splitting into an unbiased 80/20 train/test layout and tuning structural limits (`max_depth`, `min_samples_split`) to safeguard against model overfitting.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

# Define hyperparameter exploration space for optimization tracking
param_grid = {
    'max_depth': [3, 5, 7, 10],
    'min_samples_split': [2, 5, 10]
}

grid_search = GridSearchCV(estimator=DecisionTreeClassifier(random_state=42), 
                           param_grid=param_grid, 
                           cv=5, 
                           scoring='f1', 
                           n_jobs=-1)
grid_search.fit(X_train, y_train)

best_dt_model = grid_search.best_estimator_
print("--- Optimal Hyperparameter Settings Found ---")
print(grid_search.best_params_)

### 3. Model Diagnostic Metrics & Evaluation Matrices
Extracting test calculations including Confusion Matrix layouts, precision boundaries, recall records, and macro F1 scores.

In [ ]:
y_pred = best_dt_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

print("--- Confusion Matrix Breakdown ---")
print(cm)
print("\n--- Classification Performance Summary ---")
print(classification_report(y_test, y_pred, target_names=['Dissatisfied', 'Satisfied']))
print("Definitive Test F1-Score:", f1_score(y_test, y_pred))

### Required Visualizations: Model Interpretability Map & Feature Rankings
Displaying the logic boundaries of the primary levels of the optimized tree and plotting the top features driving customer sentiments.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Visualization A: Decision Tree Structural Path (Truncated for clean rendering clarity)
plot_tree(best_dt_model, max_depth=2, feature_names=list(X_encoded.columns), 
          class_names=['Dissatisfied', 'Satisfied'], filled=True, rounded=True, ax=axes[0])
axes[0].set_title('Decision Tree Split Pathways (Top Tiers)')

# Visualization B: Relative Feature Importance Allocations
importances = best_dt_model.feature_importances_
feat_importances = pd.Series(importances, index=X_encoded.columns).sort_values(ascending=False).head(10)
sns.barplot(x=feat_importances.values, y=feat_importances.index, ax=axes[1], palette='viridis')
axes[1].set_title('Top 10 Drivers of Passenger Satisfaction')
axes[1].set_xlabel('Relative Feature Importance Score')

plt.tight_layout()
plt.show()

### 4. Strategic Business Insights & Algorithmic Trade-off Analysis

#### Comparative Performance: Decision Trees vs. Logistic Regression
While Logistic Regression sets a rigid linear boundary that assumes features add up independently to predict satisfaction, our optimized Decision Tree model acts as a flexible step-function. It naturally captures non-linear relationships and complex feature interactions without needing manual transformations. For instance, the impact of a high `In-flight Wifi` score might change dramatically depending on whether the passenger is traveling for business or pleasure. The Decision Tree excels at isolating these conditional pathways, providing a more intuitive and operational look at customer segments.

#### Top Operational Drivers of Passenger Satisfaction
Looking at our feature importance rankings, factors like **In-flight Wifi Quality** and **Seat Comfort** stand out as the primary root-level splits in the tree. This indicates that operational consistency in basic cabin comfort is the single fastest way to change customer perceptions. The tree structure proves that technical metrics take a backseat to the direct passenger experience during their flight.

#### Corporate Trade-offs: False Positives vs. False Negatives
- **False Positive (Predicting a customer is Satisfied when they are actually Dissatisfied):** This is highly dangerous for customer retention. If management incorrectly assumes a group of travelers is happy, they miss the chance to reach out with targeted loyalty points or customer care recovery options, leading to unaddressed churn.
- **False Negative (Predicting a customer is Dissatisfied when they are actually Satisfied):** This leads to minor inefficiencies, such as sending customer-retention promos to someone who is already loyal. While this costs a bit of marketing budget, it doesn't cause active revenue loss.
- **Actionable Strategy:** The business should tune model thresholds to maximize the **Recall score** for the dissatisfied class. Catching every unhappy traveler is critical for long-term customer satisfaction and brand loyalty.